# PyDAP como cliente


`PyDAP` puede usarse para inspeccionar y recuperar datos remotos de forma "perezosa" desde cualquiera de los miles de datasets científicos disponibles en internet en servidores de datos [OPeNDAP](http://www.opendap.org/), permitiendo a la persona usuaria manipular un `Dataset` como si estuviera almacenado localmente, descargando sobre la marcha solo cuando sea necesario. Para transmitir datos del servidor al cliente, `tanto el servidor como el cliente deben acordar una forma de representar los datos`: **¿es un arreglo de enteros?**, **¿una malla multidimensional?** Para ello, un protocolo DAP define un **modelo de datos** que, en teoría, debería poder representar cualquier dataset (científico) existente.


Pydap usa la biblioteca `requests` para obtener datos remotos desde un servidor de datos OPeNDAP. Los datos de ese servidor son de uno de los siguientes tipos:

| Extensión de archivo | Tipo de archivo | Protocolo | URL de ejemplo |
| :- | :- | :- | -: |
| DMR | Metadatos | DAP4 | http://test.opendap.org/opendap/data/nc/coads_climatology.nc.dmr |
| DAP | Metadatos y binario |  DAP4 | http://test.opendap.org/opendap/data/nc/coads_climatology.nc.dap |
| DDS | Metadatos | DAP2  | http://test.opendap.org/opendap/data/nc/coads_climatology.nc.dds |
| DAS | Metadatos | DAP2 | http://test.opendap.org/opendap/data/nc/coads_climatology.nc.das |
| DODS | Metadatos y binario | DAP2 | http://test.opendap.org/opendap/data/nc/coads_climatology.nc.dods |

```{note}
Hacer clic en cualquiera de las URLs de ejemplo `dap` o `dods` activará una descarga de datos binarios OPeNDAP. Pydap analiza estos datos binarios y los convierte en un Dataset pydap.
```


## Requests

A partir de la versión `3.5.4`, `pydap` ahora usa `requests` para descargar archivos remotos descritos en la tabla anterior. Ademas `pydap` también puede usar `requests_cache` la cual permite persistir y reutilizar la information descargada. `Pydap` tiene una función especial para inicializar cualquiera de estas sesiones:


| Sesión sin caché | Sesión con caché |
| :- | :- |
| use_cache=False (default) | use_cache=True |


In [ ]:
from pydap.client import open_url
from pydap.net import create_session

In [ ]:
data_url = "http://test.opendap.org/opendap/data/nc/coads_climatology.nc"

## Usar sesión predeterminada sin caché


In [ ]:
# default
my_session = create_session()

In [ ]:
%%time
pyds = open_url(data_url, protocol="dap4", session=my_session)

### Intentemos de nuevo


In [ ]:
%%time
pyds = open_url(data_url, protocol="dap4", session=my_session)

### ¿Qué está pasando?


En ambos casos, solo se obtuvo el `dmr` asociado con el dataset remoto y se usó para crear el dataset pydap.


La diferencia aparente en tiempos a veces puede atribuirse a lo que se llama "lectura fría" frente a "lectura cálida". Pero en ambos escenarios, cada vez que se crea `pyds`, el dataset remoto `dmr` se obtiene y pydap lo procesa para crear el dataset `lazy` que apunta a la fuente opendap original.

Para evitar descargar repetidamente el mismo recurso una y otra vez, y potencialmente sobrecargar servidores de datos remotos, pydap ahora puede cachear respuestas.


## Usar Cached-Session


In [ ]:
# Non-default
cached_session = create_session(use_cache=True)

### Eliminar cualquier sesión previa


In [ ]:
cached_session.cache.clear()

In [ ]:
%%time
new_pyds = open_url(data_url, protocol="dap4", session=cached_session)

El tiempo requerido para descargar un `dmr` remoto desde el mismo servidor sigue siendo cercano al caso `warm`.

### Ahora intentemos de nuevo.


In [ ]:
%%time
new_pyds = open_url(data_url, protocol="dap4", session=cached_session)

El tiempo resultante se ha reducido significativamente. Esto se debe a que el `dmr` solo se descargó una vez de la fuente remota y todas las veces siguientes se reutilizo la informacion previamente descargada.


In [ ]:
print("Default location of cached response: ", cached_session.cache.db_path)

In [ ]:
print("URLs of cached responses: ", cached_session.cache.urls())

### Finalmente


In [ ]:
cached_session.cache.clear()

In [ ]:
print("URLs of cached responses: ", cached_session.cache.urls())

## Timeout

Para especificar un timeout para el cliente, simplemente establece el número deseado de segundos usando la opción `timeout` en `open_url(...)`. Por ejemplo, lo siguiente produciría timeout después de 30 segundos sin recibir una respuesta del servidor:

```{code}
dataset = open_url('http://test.opendap.org/dap/data/nc/coads_climatology.nc', timeout=30)
```

```{note}
El timeout predeterminado es 120 segundos, o 2 minutos.
```
